# ASR bằng Whisper, chia theo shot

- **Đầu vào:** video (dataset Kaggle) và file shot `<video_id>.scenes.csv` từ notebook
  `extract_scenes.ipynb`.
- **Đầu ra:** `/kaggle/working/asr/<video_id>/S<scene_id:05d>.json`
  `{video_id, scene_id, start_time_s, end_time_s, text, segments: [{start, end, text, overlap}]}`.
- **Sau đó:** chép vào `data/asr/` rồi chạy `python scripts/build_es_index.py`.

Một câu nói được gán cho mọi shot mà nó giao nhau hơn 0,1 s. Video không có tiếng (CCTV, flycam)
cho ra các shot rỗng. Bật GPU (T4 x2 hoặc P100).

In [ ]:
!pip install -q faster-whisper

## Cấu hình
Các máy được chia theo **dung lượng video** để xong gần cùng lúc.

In [ ]:
from pathlib import Path

VIDEOS_DIR = Path("/kaggle/input/datasets/dwctien/aic26-videos-batch2")  # quét đệ quy
SCENES_DIR = Path("/kaggle/input/aic26-scenes/scenes")
OUTPUT_DIR = Path("/kaggle/working/asr")
ONLY_VIDEOS = []  # rỗng = mọi video; hoặc ["M02", "M03_V001"]

SHARD_INDEX = 1  # ← đổi theo máy: 1, 2, 3 hoặc 4
NUM_SHARDS = 4

WHISPER_MODEL = "large-v3"
BEAM_SIZE = 5
MIN_OVERLAP_S = 0.1
OVERWRITE = False

## Hàm xử lý

In [ ]:
import csv
import json
import subprocess
import tempfile


def select_shard(items, shard_index, num_shards, weight=None):
    """Part shard_index (1-based) of num_shards. Items are sorted first so every machine agrees.

    Without weight: contiguous chunks. With weight: the heaviest item goes to the lightest
    shard (e.g. by video size), so all machines finish at about the same time.
    """
    assert 1 <= shard_index <= num_shards, "SHARD_INDEX phải nằm trong [1, NUM_SHARDS]"
    ordered = sorted(items)
    if num_shards == 1:
        return ordered
    if weight is None:
        size = -(-len(ordered) // num_shards)
        return ordered[(shard_index - 1) * size : shard_index * size]
    shards = [[] for _ in range(num_shards)]
    totals = [0.0] * num_shards
    for item in sorted(ordered, key=weight, reverse=True):
        lightest = totals.index(min(totals))
        shards[lightest].append(item)
        totals[lightest] += weight(item)
    return sorted(shards[shard_index - 1])


def matches_any(video_id, selectors):
    """True when no selector is given or video_id equals / starts with one of them."""
    return not selectors or any(video_id == s or video_id.startswith(s) for s in selectors)


def extract_audio(video_path, audio_path):
    """16 kHz mono WAV for Whisper; False when the video has no audio track."""
    audio_path.unlink(missing_ok=True)
    command = [
        "ffmpeg", "-i", str(video_path), "-vn", "-acodec", "pcm_s16le",
        "-ar", "16000", "-ac", "1", str(audio_path), "-y", "-loglevel", "error",
    ]  # fmt: skip
    subprocess.run(command, capture_output=True, check=False)
    return audio_path.exists() and audio_path.stat().st_size > 0


def read_shots(csv_path):
    with open(csv_path, encoding="utf-8") as f:
        return [
            {
                "video_id": row["video_id"],
                "scene_id": int(row["scene_id"]),
                "start_time_s": float(row["start_time_s"]),
                "end_time_s": float(row["end_time_s"]),
            }
            for row in csv.DictReader(f)
        ]


def map_asr_to_shots(segments, shots):
    """Attach every speech segment to the shots it overlaps by more than MIN_OVERLAP_S."""
    results = [{**shot, "text": "", "segments": []} for shot in shots]
    for seg_start, seg_end, seg_text in segments:
        if not seg_text:
            continue
        for shot in results:
            overlap = max(
                0.0, min(seg_end, shot["end_time_s"]) - max(seg_start, shot["start_time_s"])
            )
            if overlap > MIN_OVERLAP_S:
                shot["segments"].append(
                    {
                        "start": seg_start,
                        "end": seg_end,
                        "text": seg_text,
                        "overlap": round(overlap, 2),
                    }
                )
                shot["text"] = f"{shot['text']} {seg_text}".strip()
    return results


paths = {
    path.stem: path
    for pattern in ("*.mp4", "*.mkv", "*.avi", "*.mov")
    for path in VIDEOS_DIR.rglob(pattern)
    if matches_any(path.stem, ONLY_VIDEOS)
}
video_ids = select_shard(
    list(paths), SHARD_INDEX, NUM_SHARDS, weight=lambda v: paths[v].stat().st_size
)
print(f"Shard {SHARD_INDEX}/{NUM_SHARDS}: {len(video_ids)}/{len(paths)} video")

## Chạy Whisper
`vad_filter` ngăn Whisper tự bịa chữ (ví dụ "Hãy đăng ký kênh") trên đoạn im lặng;
`condition_on_previous_text=False` tránh lặp mãi một câu sai. Chạy lại cell để tiếp tục.

In [ ]:
from faster_whisper import WhisperModel
from tqdm.auto import tqdm

model = WhisperModel(WHISPER_MODEL, device="cuda", compute_type="float16")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
failed = {}

with tempfile.TemporaryDirectory() as tmp:
    audio_path = Path(tmp) / "audio.wav"
    for video_id in tqdm(video_ids, desc="ASR"):
        csv_path = SCENES_DIR / f"{video_id}.scenes.csv"
        video_out = OUTPUT_DIR / video_id
        if not csv_path.exists():
            failed[video_id] = f"không có {csv_path.name}"
            continue
        if video_out.is_dir() and any(video_out.iterdir()) and not OVERWRITE:
            continue
        segments = []
        if extract_audio(paths[video_id], audio_path):
            try:
                raw_segments, _ = model.transcribe(
                    str(audio_path),
                    language="vi",
                    beam_size=BEAM_SIZE,
                    vad_filter=True,
                    vad_parameters={"min_silence_duration_ms": 500},
                    condition_on_previous_text=False,
                )
                segments = [(s.start, s.end, s.text.strip()) for s in raw_segments]
            except Exception as exc:  # noqa: BLE001 - một video lỗi không được dừng cả lô
                failed[video_id] = f"Whisper lỗi: {exc}"
                continue
        video_out.mkdir(parents=True, exist_ok=True)
        for shot in map_asr_to_shots(segments, read_shots(csv_path)):
            (video_out / f"S{shot['scene_id']:05d}.json").write_text(
                json.dumps(shot, ensure_ascii=False, indent=4), encoding="utf-8"
            )

print(f"Xong {len(video_ids) - len(failed)}/{len(video_ids)} video")
for video_id, error in failed.items():
    print(f"[LỖI] {video_id}: {error}")

## Nén kết quả
Tải file nén ở tab **Output** của Kaggle rồi giải nén vào `data/asr/` trên máy build.

In [ ]:
import shutil

archive = shutil.make_archive(
    str(OUTPUT_DIR.parent / f"{OUTPUT_DIR.name}_shard{SHARD_INDEX}of{NUM_SHARDS}"),
    "zip",
    root_dir=str(OUTPUT_DIR.parent),
    base_dir=OUTPUT_DIR.name,
)
print(f"{archive}: {Path(archive).stat().st_size / 2**20:.1f} MB")